# Detección en vivo local: FastRTC + YOLOv8 + Supervision

Versión para ejecutar localmente desde **Google Antigravity IDE en Windows**.

`webcam → WebRTC local → YOLOv8 → Supervision → navegador`

Al ejecutarse en el mismo equipo mediante `127.0.0.1`, no necesita Colab, TURN, Twilio, Hugging Face Token ni un enlace público.

## 1. Crear y seleccionar el entorno de Python

En la terminal de Antigravity, dentro de esta carpeta, ejecuta una sola vez:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip ipykernel
python -m ipykernel install --user --name fastrtc-local --display-name "Python (FastRTC Local)"
```

Luego abre el selector de kernel del notebook y elige **Python (FastRTC Local)**. Se recomienda Python 3.11 o 3.12.

In [ ]:
# Instala todo en el kernel seleccionado. FastRTC instala Gradio y aiortc.
%pip install -q "fastrtc==0.0.34" ultralytics supervision

> Si alguno de estos paquetes ya estaba importado antes de la instalación, reinicia solamente el kernel del notebook. Después continúa sin repetir esta celda.

In [ ]:
import sys
from importlib.metadata import version

import numpy as np
import supervision as sv
import torch
from fastrtc import Stream, VideoStreamHandler
from ultralytics import YOLO

print("Python ejecutable:", sys.executable)
print("Python versión:", sys.version.split()[0])
for package in ("fastrtc", "gradio", "ultralytics", "supervision", "aiortc"):
    print(f"{package}: {version(package)}")
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Cargar YOLOv8 y Supervision

Se conserva `yolov8n.pt` para mantener el mismo modelo de la clase original. La `n` significa *nano*: es apropiado para webcam y también puede ejecutarse en CPU. La primera ejecución descarga los pesos.

In [ ]:
MODEL_NAME = "yolov8n.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(MODEL_NAME)
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
print(f"Modelo: {MODEL_NAME} | dispositivo: {DEVICE}")

## 3. Procesar cada frame

FastRTC entrega un arreglo NumPy. Ultralytics detecta objetos y Supervision convierte el resultado y dibuja las cajas y etiquetas.

In [ ]:
CONFIDENCE = 0.30
IMAGE_SIZE = 640

def process_frame(frame: np.ndarray) -> np.ndarray:
    if frame is None:
        return frame

    result = model.predict(
        source=frame,
        conf=CONFIDENCE,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        verbose=False,
    )[0]

    detections = sv.Detections.from_ultralytics(result)
    annotated = box_annotator.annotate(
        scene=frame.copy(), detections=detections
    )

    if detections.class_id is not None and detections.confidence is not None:
        labels = [
            f"{model.names[int(class_id)]} {confidence:.2f}"
            for class_id, confidence in zip(
                detections.class_id, detections.confidence
            )
        ]
        annotated = label_annotator.annotate(
            scene=annotated, detections=detections, labels=labels
        )

    return annotated

## 4. Iniciar el WebRTC local

Detén cualquier instancia anterior que siga usando el puerto de Gradio. Esta celda abre automáticamente la aplicación local en el navegador. `skip_frames=True` evita acumular retraso cuando el modelo no alcanza los FPS de la cámara.

In [ ]:
stream = Stream(
    handler=VideoStreamHandler(process_frame, skip_frames=True),
    modality="video",
    mode="send-receive",
    rtc_configuration=None,
    concurrency_limit=1,
    time_limit=300,
    ui_args={
        "hide_title": True,
        "full_screen": False,
    },
)

stream.ui.launch(
    server_name="127.0.0.1",
    share=False,
    inbrowser=True,
    debug=True,
)

## Diagnóstico local

- **No abre la página:** mira la URL impresa por Gradio y ábrela manualmente. Si `7860` está ocupado, Gradio normalmente seleccionará otro puerto.
- **La cámara no aparece:** concede permiso de cámara a `127.0.0.1` en el navegador y cierra otras aplicaciones que la estén usando.
- **CUDA aparece como `False`:** el notebook funcionará con CPU. Para NVIDIA, instala una compilación de PyTorch compatible con el controlador de tu equipo.
- **El video tiene retraso:** conserva `skip_frames=True`, cambia `IMAGE_SIZE` a `480` o usa GPU.
- **Quieres abrirlo desde otro equipo:** `localhost` solo funciona en esta computadora; una publicación remota vuelve a requerir HTTPS y TURN.